In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [19]:
df = pd.read_csv('cleaned_customer_data.csv')
print(df.head(20))

    customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0   7590-VHVEG  Female              0     Yes         No       1           No   
1   5575-GNVDE    Male              0      No         No      34          Yes   
2   3668-QPYBK    Male              0      No         No       2          Yes   
3   7795-CFOCW    Male              0      No         No      45           No   
4   9237-HQITU  Female              0      No         No       2          Yes   
5   9305-CDSKC  Female              0      No         No       8          Yes   
6   1452-KIOVK    Male              0      No        Yes      22          Yes   
7   6713-OKOMC  Female              0      No         No      10           No   
8   7892-POOKP  Female              0     Yes         No      28          Yes   
9   6388-TABGU    Male              0      No        Yes      62          Yes   
10  9763-GRSKD    Male              0     Yes        Yes      13          Yes   
11  7469-LKBCI    Male      

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ========== DATA PREPROCESSING ==========
if df['Churn'].dtype == 'object':
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 
                      'numAdminTickets', 'numTechTickets']

categorical_features = ['gender', 'Partner', 'Dependents', 'Contract', 
                        'PaperlessBilling', 'InternetService']

X = df[numerical_features + categorical_features].copy()
y = df['Churn']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Encode categorical variables
for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Scale numerical features
scaler = StandardScaler()
X[numerical_features] = scaler.fit_transform(X[numerical_features])

print(" Preprocessing done!")

# ========== TRAIN TEST SPLIT ==========
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

# ========== MODEL TRAINING ==========
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100)
}

print("\n" + "="*60)
print("MODEL COMPARISON RESULTS")
print("="*60)

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    print(f"\n{name}:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

# ========== FIND BEST MODEL (JO BHI BEST HAI) ==========
best_result = max(results, key=lambda x: x['F1-Score'])
best_model_name = best_result['Model']

print("\n" + "="*60)
print(f"🏆 BEST MODEL: {best_model_name}")
print("="*60)

# ========== TRAIN BEST MODEL (LOGISTIC REGRESSION) ==========
if best_model_name == 'Logistic Regression':
    best_model = LogisticRegression(random_state=42)
elif best_model_name == 'Decision Tree':
    best_model = DecisionTreeClassifier(random_state=42, max_depth=5)
else:
    best_model = RandomForestClassifier(random_state=42, n_estimators=100)

best_model.fit(X_train, y_train)
final_predictions = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, final_predictions)

print("\n" + "="*60)
print(f"✅ BEST MODEL ({best_model_name}) ACCURACY: {final_accuracy:.2%}")
print("="*60)

# ========== PREDICTION RESULTS ==========
print("\n" + "="*60)
print(" PREDICTION RESULTS ON TEST DATA")
print("="*60)

print(f"\n Total Test Customers: {len(X_test)}")
print(f"   - Actually Churn: {(y_test == 1).sum()}")
print(f"   - Actually No Churn: {(y_test == 0).sum()}")

# Sample predictions (first 20)
print("\n SAMPLE PREDICTIONS (First 20 Test Customers)")
print("-" * 60)

for i in range(20):
    actual = "CHURN" if y_test.iloc[i] == 1 else "NO"
    predicted = "CHURN" if final_predictions[i] == 1 else "NO"
    status = "✓" if actual == predicted else "✗"
    print(f"Customer {i+1:2d}: Actual={actual:4} | Predicted={predicted:4} | {status}")

# Confusion Matrix 
cm = confusion_matrix(y_test, final_predictions)

print("\n" + "="*60)
print(" CONFUSION MATRIX")
print("="*60)
print(f" Correct No Churn:  {cm[0][0]}")
print(f" Wrong Churn:       {cm[0][1]}")
print(f" Missed Churn:      {cm[1][0]}")
print(f" Correct Churn:     {cm[1][1]}")

# Final Summary
correct = (final_predictions == y_test).sum()
wrong = len(y_test) - correct

print("\n" + "="*60)
print(" FINAL SUMMARY")
print("="*60)
print(f" Correct Predictions: {correct}/{len(y_test)} ({final_accuracy:.2%})")
print(f" Wrong Predictions: {wrong}/{len(y_test)} ({1-final_accuracy:.2%})")

print("\n" + "="*60)
print(" PREDICTION COMPLETED SUCCESSFULLY!")
print("="*60)


Features shape: (7032, 11)
Target shape: (7032,)
 Preprocessing done!

Training set: 5625 rows
Test set: 1407 rows

MODEL COMPARISON RESULTS

Logistic Regression:
  Accuracy:  0.8422
  Precision: 0.7184
  Recall:    0.6684
  F1-Score:  0.6925

Decision Tree:
  Accuracy:  0.8387
  Precision: 0.7394
  Recall:    0.6070
  F1-Score:  0.6667

Random Forest:
  Accuracy:  0.8365
  Precision: 0.7118
  Recall:    0.6471
  F1-Score:  0.6779

🏆 BEST MODEL: Logistic Regression

✅ BEST MODEL (Logistic Regression) ACCURACY: 84.22%

 PREDICTION RESULTS ON TEST DATA

 Total Test Customers: 1407
   - Actually Churn: 374
   - Actually No Churn: 1033

 SAMPLE PREDICTIONS (First 20 Test Customers)
------------------------------------------------------------
Customer  1: Actual=NO   | Predicted=NO   | ✓
Customer  2: Actual=NO   | Predicted=CHURN | ✗
Customer  3: Actual=NO   | Predicted=NO   | ✓
Customer  4: Actual=CHURN | Predicted=NO   | ✗
Customer  5: Actual=NO   | Predicted=NO   | ✓
Customer  6: Actual